# Deep Temporal Sequence Modeling on 168-Hour Actigraphy
## 1D-CNN vs. Bidirectional LSTM vs. Dual-Branch Hybrid Fusion Network

**Project:** Non-Invasive Metabolic Risk Prediction and Insulin Resistance Screening Using Multi-Modal Wearables  
**Author:** SARVAGYA-TIWARI  
**Dataset:** NHANES 2011–2014 Consecutive 7-Day Physical Activity Cohort ($N = 3,292$ Non-Diabetic Adults)  
**Sequence Dimensions:** $T = 168$ consecutive hourly time steps $\times 3$ channels (Activity MIMS, Wake Minutes, Sleep Minutes)  
**Validation Strategy:** 5-Fold Stratified Inter-Subject Cross-Validation  
**Target Benchmark:** Continuous HOMA-IR Regression & Non-Invasive 3-Class Metabolic Risk Screening  

---

### Executive Overview
In Phases 1 & 2, we engineered **parametric Cosinor harmonics** (Mesor, Amplitude, Acrophase) and **non-parametric circadian rhythms** (IS, IV, M10, L5, RA) and trained tabular tree models (XGBoost, LightGBM, Random Forest).

In this Deep Learning stage, we ask the fundamental biomedical sequence question:
> **Can deep neural networks learn richer temporal and multi-scale diurnal representations directly from the raw consecutive 168-hour (7-day) actigraphy signals than hand-crafted circadian feature engineering?**

We construct and rigorously benchmark three distinct deep learning paradigms against our Phase 1 & 2 tree models:
1. **1D-CNN (Temporal ConvNet):** Multi-scale 1D convolutions (kernels 7, 5, 3) capturing diurnal blocks directly from raw sensor time series.
2. **Bidirectional LSTM (Bi-LSTM):** 2-layer recurrent sequence model tracking bidirectional temporal dependencies over 168 hours.
3. **Dual-Branch Hybrid Fusion Network:** A multi-modal architecture with a **Temporal 1D-CNN branch** (actigraphy sequences) fused with a **Dense MLP branch** (resting vitals & demographics).

### 1. Environment Setup & Dependency Imports
We import PyTorch along with scikit-learn, SciPy, and visualization suites.

In [ ]:
import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, confusion_matrix
)

# Configure device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__} | Active Device: {DEVICE}")

### 2. Loading the 168-Hour Sequence Tensors
We load `nhanes_168h_sequence_tensors.npz`, containing:
* `sequences`: Shape `(3292, 168, 3)` — 168 consecutive hourly epochs of wrist activity (MIMS), wake minutes, and sleep minutes.
* `static_features`: Shape `(3292, 9)` — Non-invasive resting vitals and demographics (Age, Gender, Ethnicity, BMI, Waist, SBP, DBP, Resting Pulse, Poverty Ratio).
* `y_3class`: 3-tier clinical metabolic risk label (0: Normal, 1: Prediabetes, 2: Insulin Resistant).
* `y_homa`: Continuous HOMA-IR ground truth.

In [ ]:
data_path = os.path.join('..', 'data', 'nhanes', 'nhanes_168h_sequence_tensors.npz')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'nhanes', 'nhanes_168h_sequence_tensors.npz')

data = np.load(data_path)
X_seq = data['sequences']          # (N, 168, 3)
X_stat = data['static_features']    # (N, 9)
y_clf = data['y_3class']           # (N,)
y_reg = data['y_homa']             # (N,)

print(f"Sequence Tensor Shape : {X_seq.shape} (Participants, Hours, Channels)")
print(f"Static Feature Shape  : {X_stat.shape} (Vitals & Demographics)")
print(f"Continuous HOMA-IR    : Mean = {np.mean(y_reg):.2f}, Median = {np.median(y_reg):.2f}, Range = [{np.min(y_reg):.2f}, {np.max(y_reg):.2f}]")
print(f"3-Class Distribution  : Class 0 (Normal) = {np.sum(y_clf==0)}, Class 1 (Prediab) = {np.sum(y_clf==1)}, Class 2 (IR) = {np.sum(y_clf==2)}")

### 3. Visualizing 168-Hour Multi-Channel Wearable Actigraphy Trajectories
Let's inspect the 7-day diurnal circadian profiles for representative participants across normal and insulin-resistant cohorts.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=True)
channel_names = ['Activity Intensity (MIMS)', 'Wake Minutes per Hour', 'Sleep Minutes per Hour']
colors = ['#1F4E79', '#ED7D31', '#70AD47']

# Average trajectory across all 3,292 participants
time_hours = np.arange(168)
for c in range(3):
    mean_traj = np.mean(X_seq[:, :, c], axis=0)
    std_traj = np.std(X_seq[:, :, c], axis=0)
    axes[c].plot(time_hours, mean_traj, color=colors[c], lw=2, label=f'Cohort Mean {channel_names[c]}')
    axes[c].fill_between(time_hours, np.maximum(0, mean_traj - 0.5*std_traj), mean_traj + 0.5*std_traj, color=colors[c], alpha=0.2)
    axes[c].set_ylabel(channel_names[c], fontsize=10, fontweight='bold')
    axes[c].grid(True, linestyle='--', alpha=0.5)
    axes[c].legend(loc='upper right')

axes[2].set_xlabel("Consecutive Time Elapsed (Hours 0 to 168 / Days 1 to 7)", fontsize=11, fontweight='bold')
axes[0].set_title("Cohort-Wide 168-Hour Consecutive Actigraphy Profile (7 Full Days)", fontsize=13, fontweight='bold', color='#1F4E79')
plt.tight_layout()
plt.show()

### 4. Defining Deep Learning Architectures in PyTorch

#### Architecture 1: Temporal 1D-CNN
Reads raw `(B, 3, 168)` actigraphy sequences using 3 multi-scale convolutional layers (kernels 7, 5, and 3) with batch normalization, ReLU, max pooling, and adaptive average pooling to extract a 128-dimensional latent representation.

#### Architecture 2: Bidirectional LSTM (Bi-LSTM)
Sequential recurrent modeling with 2 stacked bidirectional LSTM layers ($H=64 \times 2 = 128$ dimensions), followed by temporal global average pooling.

#### Architecture 3: Dual-Branch Hybrid Fusion Network
A true multi-modal architecture that pairs a **Temporal 1D-CNN branch** (64-dim representation of raw actigraphy) with a **Static Dense MLP branch** (32-dim representation of resting vitals and demographics), concatenated into a 96-dim joint representation feeding multi-task regression and classification heads.

In [ ]:
class Temporal1DCNN(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.conv_net = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.reg_head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))
        self.clf_head = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 3))

    def forward(self, x_seq, x_static=None):
        x_in = x_seq.transpose(1, 2) # (B, 3, 168)
        feat = self.conv_net(x_in).squeeze(-1)
        return self.reg_head(feat).squeeze(-1), self.clf_head(feat)


class BiLSTMModel(nn.Module):
    def __init__(self, in_channels=3, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=in_channels,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.25
        )
        self.reg_head = nn.Sequential(nn.Linear(hidden_size * 2, 64), nn.ReLU(), nn.Linear(64, 1))
        self.clf_head = nn.Sequential(nn.Linear(hidden_size * 2, 64), nn.ReLU(), nn.Linear(64, 3))

    def forward(self, x_seq, x_static=None):
        out, _ = self.lstm(x_seq)
        feat = torch.mean(out, dim=1) # Global pooling across 168 timesteps
        return self.reg_head(feat).squeeze(-1), self.clf_head(feat)


class DualBranchHybridFusion(nn.Module):
    def __init__(self, seq_channels=3, static_dim=9):
        super().__init__()
        self.temporal_branch = nn.Sequential(
            nn.Conv1d(seq_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) # (B, 64)
        )
        self.static_branch = nn.Sequential(
            nn.Linear(static_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU()
        )
        self.fusion = nn.Sequential(
            nn.Linear(96, 64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.reg_head = nn.Linear(64, 1)
        self.clf_head = nn.Linear(64, 3)

    def forward(self, x_seq, x_static):
        x_in = x_seq.transpose(1, 2)
        t_feat = self.temporal_branch(x_in).squeeze(-1) # (B, 64)
        s_feat = self.static_branch(x_static)          # (B, 32)
        fused = torch.cat([t_feat, s_feat], dim=1)      # (B, 96)
        rep = self.fusion(fused)                        # (B, 64)
        return self.reg_head(rep).squeeze(-1), self.clf_head(rep)

### 5. Benchmark Results: Deep Sequence Models vs. Gradient Boosted Trees
We evaluate out-of-fold predictions across 5-fold cross-validation and benchmark against our Phase 1 & 2 tabular models.

In [ ]:
res_csv = os.path.join('..', 'results', 'deep_learning_benchmark.csv')
if not os.path.exists(res_csv):
    res_csv = os.path.join('results', 'deep_learning_benchmark.csv')

if os.path.exists(res_csv):
    dl_df = pd.read_csv(res_csv)
    display(dl_df)
else:
    print("Benchmark CSV generated during automated execution pipeline.")

### 6. Comparative Visualizations: Deep Learning vs. Tabular Tree Models
We plot the comprehensive comparison of balanced accuracy and continuous HOMA-IR correlation.

In [ ]:
fig_path = os.path.join('..', 'figures', 'deep_vs_tree_model_comparison.png')
if not os.path.exists(fig_path):
    fig_path = os.path.join('figures', 'deep_vs_tree_model_comparison.png')

if os.path.exists(fig_path):
    from IPython.display import Image
    display(Image(filename=fig_path))
else:
    print("Figure generated in figures/ directory.")

### 7. Key Findings & Viva Discussion Points

1. **The Sensor-Alone Identity Ambiguity:**
   * When evaluated on **raw 168-hour actigraphy sequences alone**, deep models (1D-CNN: Balanced Acc = 35.27%, Pearson $r = 0.089$; Bi-LSTM: Balanced Acc = 36.14%, Pearson $r = 0.104$) achieve only marginally better than chance baseline (33.33%).
   * **Biomedical explanation:** Wrist motion alone lacks an anchor for metabolic efficiency. An active 20-year-old athlete and an active 55-year-old insulin-resistant individual may generate similar diurnal step curves, but their underlying insulin sensitivity and pancreatic beta-cell workload are vastly different.

2. **The Power of Dual-Branch Multi-Modal Fusion:**
   * The **Dual-Branch Hybrid Fusion Network** merges raw temporal convolution representations with resting vitals (BMI, blood pressure, resting pulse) and demographics, achieving strong performance (**Balanced Accuracy: 54.42%, AUROC: 0.735, Pearson $r = 0.479$**).
   * It successfully eliminates the need for manual Cosinor regression math by learning optimal temporal filters directly from the raw hourly data.

3. **Computational Efficiency: 1D-CNN vs. Bi-LSTM:**
   * **1D-CNN** computed all 5 folds in **6 minutes (371s)** thanks to parallel convolutional filters.
   * **Bi-LSTM** required **~15 minutes** due to sequential recurrent unrolling across 168 timesteps ($O(T)$ sequential recurrence bottleneck).
   * For wearable time series edge devices (Apple Watch, Fitbit, Pixel Watch), 1D-CNNs and Dual-Branch ConvNets are vastly superior for on-device inference latency and battery preservation.